# Price Baselines

How do you know a model has learned anything? You compare it against something
that has learned nothing.

This notebook builds two deliberately stupid predictors first, so that every
real model afterwards has a number to beat:

| Predictor | Knows | Roughly |
|---|---|---|
| `random_pricer` | nothing at all | what failure looks like |
| `constant_pricer` | one fact about the dataset | the bar to clear |
| `linear_regression_pricer` | a little about each item | the first real attempt |

Every one of them is scored by the same `evaluate` harness, on the same held-out
items, with the same charts. That is what makes the numbers comparable.

**Reading the output:** the error is a distance in dollars, so **lower is
better**. `r²` is a score, so higher is better -- and it is measured against the
constant baseline, which sits at exactly 0%.

## Imports

In [ ]:
import random

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

from notebooks.helpers.evaluator import evaluate
from notebooks.models.items import Item

## Load the data

Three splits come back. The models learn from `train`; they are scored on
`test`, which they never see. `validation` is for tuning later -- nothing in
this notebook tunes anything, so it goes unused.

In [ ]:
DATASET = "ed-donner/items_raw_lite"

train, validation, test = Item.from_hub(DATASET)

print(f"train={len(train):,} validation={len(validation):,} test={len(test):,}")
print(train[0])

## Baseline 1 -- what failure looks like

`random_pricer` ignores the item entirely and picks a number out of the air. It
is not trying to be good. It is here so that you know what a useless model looks
like on these charts before you have anything to compare against -- a shapeless
red cloud, and an error around $300.

It also smoke-tests the harness: if this runs and charts cleanly, the plumbing
works, and anything odd later belongs to the model rather than the measurement.

In [ ]:
def random_pricer(item):
    return random.randrange(1, 1000)


random.seed(42)
evaluate(random_pricer, test)

## Baseline 2 -- the bar to beat

`constant_pricer` also ignores the item, but it knows one thing: the average
price across the training set. That single fact is the most you can extract
without ever looking at the product.

This is the real baseline. Guessing the mean is provably the best you can do
with zero features, which is why `r²` for this predictor is exactly 0% -- every
r² printed from here on says how far a model got from *this* toward perfect.

Note the average comes from `train`, not `test`. Taking it from the data you are
scored on would be leakage, and would flatter the baseline unfairly.

In [ ]:
training_average = sum(item.price for item in train) / len(train)
print(f"training average = ${training_average:,.2f}")


def constant_pricer(item):
    return training_average


evaluate(constant_pricer, test)

## A model that actually reads the item

Three cheap features. `full` is the raw product text, present on all three
splits -- `summary` only exists on `train`, so using it here would mean training
and testing on different kinds of text.

`weight` is optional on `Item`, so a missing weight becomes `0.0` plus a flag
saying it was missing. Without the flag, the model would read "unknown" as
"weighs nothing".

In [ ]:
FEATURE_COLUMNS = ["weight", "weight_unknown", "text_length"]


def get_features(item: Item) -> dict[str, float]:
    weight = item.weight or 0.0
    return {
        "weight": weight,
        "weight_unknown": 1.0 if weight == 0 else 0.0,
        "text_length": float(len(item.full or "")),
    }


def to_dataframe(items) -> pd.DataFrame:
    frame = pd.DataFrame([get_features(item) for item in items])
    frame["price"] = [item.price for item in items]
    return frame


train_df = to_dataframe(train)
test_df = to_dataframe(test)
train_df.head()

In [ ]:
np.random.seed(42)

model = LinearRegression()
model.fit(train_df[FEATURE_COLUMNS], train_df["price"])

for feature, coefficient in zip(FEATURE_COLUMNS, model.coef_, strict=True):
    print(f"{feature}: {coefficient:.4f}")
print(f"intercept: {model.intercept_:.4f}")

In [ ]:
def linear_regression_pricer(item):
    features = pd.DataFrame([get_features(item)])[FEATURE_COLUMNS]
    return max(0.0, model.predict(features)[0])


evaluate(linear_regression_pricer, test)

## Reading the charts

**Error trend** -- the running average error, with a 95% confidence band. If the
band is still wide at the last datapoint, two models a few dollars apart are not
actually distinguishable; raise `size` before believing the gap.

**Scatter** -- one dot per item, guess against truth, with the dashed `y = x`
line. Dots hugging the line are good guesses. A flat horizontal band means the
predictor answers the same thing whatever it is shown, which is exactly what
`constant_pricer` looks like.

## Extending this

`evaluate` takes any callable of one `Item` that returns a price. Nothing about
the harness needs to change to add a model -- write the function, call
`evaluate`, compare the number against the constant baseline:

```python
def my_pricer(item):
    return some_prediction(item)

evaluate(my_pricer, test)
```

Strings are fine too, so an LLM answering `"about $1,249.99"` scores the same
way as a model returning a float. Useful knobs:

```python
evaluate(my_pricer, test, size=500)      # more datapoints, tighter confidence band
evaluate(my_pricer, test, workers=20)    # more threads, for API-bound predictors
tester = evaluate(my_pricer, test)       # returns the Tester
worst = sorted(tester.results, key=lambda result: -result.error)[:10]
```

Sensible next rungs on the ladder: bag-of-words over `full` with
`CountVectorizer`, then a `RandomForestRegressor` on those vectors, then an LLM.